# WASP-52b H$\alpha$: LaRT-coupled 2D (cylindrical Ly$\alpha$) transit

Proper H$\alpha$ transit transmission for WASP-52b, coupling the EXHALE escape
model to the **LaRT** Monte-Carlo Ly$\alpha$ radiative transfer (spherical
illumination). Under stellar illumination the Ly$\alpha$ scattering rate
$P_\alpha(\rho,z)$ -- hence the $n_{2p}$ population that absorbs H$\alpha$ -- is
only *cylindrically* symmetric about the star-planet axis, so the spherical
EXHALE_transit.py is replaced here by the 2-D integrator `examples/tpm_halpha_lart2d.py`.

Normalization: $P_\alpha = \mathrm{Pa_{LaRT}}\times L_{\rm Ly\alpha}\times
\texttt{fluxfac}$ (LaRT outputs are per unit luminosity; `fluxfac` $=\Omega_\star/4\pi$
is the geometric factor, written to the FITS/HDF5 header and *not* included in the
raw scattering-rate accumulation).

Two domains are compared: the extended **10 $R_p$** run (matching Yan 2022) and the
physical **$L_1$-truncated** run ($\simeq2.53\,R_p$).

In [1]:
import os, sys, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('../examples'))
import tpm_halpha_lart2d as h2d

OBS = '/home/kiseon/Exoplanetary_Atmosphere/WASP-52b/WASP52b_Halpha_transpec_0p10_AA_binned.txt'
obs = np.loadtxt(OBS)            # lam_air, TS (negative dip), e_TS, model

# (label, EXHALE run dir, LaRT h5, color)
cases = [
    (r'10 $R_p$, $\sigma$=49',   'fxuv0p25_he98',    'LaRT_lya/WASP-52b_lya.h5',                  'C0'),
    (r'$L_1$ (Roche), $\sigma$=49','fxuv0p25_he98_L1', 'LaRT_lya_L1/WASP-52b_L1_lya.h5',            'C1'),
    (r'$L_1$ (Roche), FWHM=49',   'fxuv0p25_he98_L1', 'LaRT_lya_L1_fwhm/WASP-52b_L1_fwhm_lya.h5',  'C2'),
]

## Incident stellar Ly$\alpha$ line profile

The Ly$\alpha$ photons incident on the planet follow the Huang et al. (2017) /
Yan et al. (2022) double-Gaussian: peaks at $\pm74$ km/s with width 49 km/s.
The width can be read as the Gaussian $\sigma$ (broad, shallow central dip) or as
the FWHM ($\sigma=20.8$ km/s; narrow peaks, deep self-reversal).

In [2]:
import exhale_to_lart as e2l
c_kms = 299792.458; lam0 = 1215.6701
lam_s, p_s = e2l.double_gaussian_lya(74, 49, nwav=601, width_is_fwhm=False)
lam_f, p_f = e2l.double_gaussian_lya(74, 49, nwav=601, width_is_fwhm=True)
v_s = (lam_s - lam0)/lam0*c_kms
v_f = (lam_f - lam0)/lam0*c_kms
fig, ax = plt.subplots(figsize=(7, 4.4))
ax.plot(v_s, p_s, 'C0-',  lw=2, label=r'width $=\sigma=49$ km s$^{-1}$')
ax.plot(v_f, p_f, 'C1--', lw=2, label=r'width $=$ FWHM $=49$ km s$^{-1}$ ($\sigma=20.8$)')
for vp in (-74, 74):
    ax.axvline(vp, color='0.7', ls=':', lw=1)
ax.set_xlim(-250, 250)
ax.set_xlabel(r'velocity [km s$^{-1}$] from Ly$\alpha$ 1215.67 \AA')
ax.set_ylabel('normalized incident flux')
ax.set_title(r'Incident stellar Ly$\alpha$ profile (double-Gaussian)')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig('lya_profile.png', dpi=140); plt.show()

In [3]:
fig, ax = plt.subplots(figsize=(8, 5.2))
# observation as transit depth (negative = absorption): TS (native)
ax.errorbar(obs[:,0], obs[:,1]*100.0, yerr=obs[:,2]*100.0, fmt='o', ms=3,
            color='0.4', ecolor='0.75', capsize=2, label='obs (binned)', zorder=1)
for lab, rundir, h5, col in cases:
    if not os.path.exists(h5):
        print('skip (LaRT output not ready):', h5); continue
    lam, Tl, info = h2d.compute_halpha(rundir, h5)
    ax.plot(lam, -(1.0 - Tl)*100.0, color=col, lw=2,
            label='%s : %.2f%%' % (lab, info['depth_pct']), zorder=3)
    print('%-22s depth=%.3f%%  P_alpha_max=%.2f s^-1  n2p_max=%.2e  fluxfac=%.3e'
          % (lab, info['depth_pct'], info['Palpha_max'], info['n2p_max'], info['fluxfac']))
ax.axhline(0, color='0.6', lw=0.8)
ax.set_xlim(6561, 6564.6)
ax.set_xlabel(r'wavelength [\AA] (air)')
ax.set_ylabel(r'transit depth [\%]  (negative $=$ absorption)')
ax.set_title(r'WASP-52b H$\alpha$: EXHALE $+$ LaRT 2D (cylindrical Ly$\alpha$)')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('halpha_lart2d.png', dpi=140)
plt.show()

10 $R_p$, $\sigma$=49  depth=1.456%  P_alpha_max=60.43 s^-1  n2p_max=2.15e+04  fluxfac=1.211e-02


$L_1$ (Roche), $\sigma$=49 depth=1.282%  P_alpha_max=85.84 s^-1  n2p_max=4.19e+04  fluxfac=7.658e-04


$L_1$ (Roche), FWHM=49 depth=1.504%  P_alpha_max=82.27 s^-1  n2p_max=1.79e+04  fluxfac=7.658e-04


## H$\alpha$ vs incident Ly$\alpha$ flux ($F_{\rm Ly\alpha}$ scan)

H$\alpha$ is Ly$\alpha$-pump-dominated, so the depth scales with the incident
Ly$\alpha$ flux. This needs **no new LaRT run**: $P_\alpha=\mathrm{Pa}\times
L_{\rm Ly\alpha}\times\texttt{fluxfac}$, so only $L_{\rm Ly\alpha}$ is rescaled
(\texttt{lya\_scale}). Yan's $F_{\rm Ly\alpha}=37{,}300$ erg\,cm$^{-2}$\,s$^{-1}$
is an EP\,Eri proxy (WASP-52's Ly$\alpha$ is unobserved); the observed H$\alpha$
then constrains the effective $F_{\rm Ly\alpha}$.

In [4]:
scales = [1.0, 0.7, 0.5, 0.3, 0.2, 0.1, 0.05]
fig, ax = plt.subplots(figsize=(7, 4.8))
for lab, rundir, h5, col in cases:
    if not os.path.exists(h5):
        continue
    deps = [h2d.compute_halpha(rundir, h5, lya_scale=s)[2]['depth_pct'] for s in scales]
    ax.plot(scales, [-d for d in deps], 'o-', color=col, lw=2, label=lab)
    print('%-22s' % lab, '  '.join('%.3f' % d for d in deps))
ax.axhspan(-1.0, -0.7, color='0.8', alpha=0.6, label='obs $\\sim$0.7--1\\%')
ax.set_xscale('log')
ax.set_xlabel(r'$F_{\rm Ly\alpha}\,/\,F_{\rm Ly\alpha}^{\rm Yan}$ (37,300 erg cm$^{-2}$ s$^{-1}$)')
ax.set_ylabel(r'H$\alpha$ line-center transit depth [\%]')
ax.set_title(r'WASP-52b H$\alpha$ vs incident Ly$\alpha$ flux (LaRT 2D)')
ax.legend(); ax.grid(alpha=0.3, which='both')
fig.tight_layout(); fig.savefig('halpha_flya_scan.png', dpi=140); plt.show()

10 $R_p$, $\sigma$=49  1.456  1.232  1.057  0.845  0.711  0.518  0.348


$L_1$ (Roche), $\sigma$=49 1.282  1.084  0.925  0.720  0.578  0.373  0.219


$L_1$ (Roche), FWHM=49 1.504  1.278  1.094  0.853  0.687  0.444  0.262
